In [1]:
# 1. registry and slope mask
import numpy as np, rasterio

In [2]:
FILES = {   # feature rasters + target, all on the 5965x5773 grid
    "elev":      "../data/rasters/CanFelCas_2000filled_5m.tif",
    "slope":     "../data/rasters/CFC_2000slope_5m.tif",
    "aspect":    "../data/rasters/CFC_2000aspect_5m.tif",
    "roughness": "../data/rasters/CFC_2000roughness_5m.tif",
    "tri":       "../data/rasters/CFC_2000TRI_5m.tif",
    "tpi":       "../data/rasters/CFC_2000TPI_5m.tif",
    "ndvi":      "../data/rasters/ndvi2000_5m_aligned.tif",
    "lst":       "../data/rasters/lst2000_5m_aligned.tif",
    "dist":      "../data/rasters/cfc2000_dist_to_basin5m.tif",
    "ddem":      "../data/rasters/cfc_ddem_2010_2000_5m.tif",   # <- target, not a feature
}
SLOPE_ID = "../data/rasters/cfc2000_slopeid_5m.tif"

with rasterio.open(SLOPE_ID) as src:
    sid = src.read(1)
    transform = src.transform
    H, W = sid.shape

rows, cols = np.where(sid>0) # the six slopes
print("grid", (H,W), "| slope pixels", rows.size)

grid (5773, 5965) | slope pixels 2605575


In [3]:
# 2. sample all rasters at slope pixels, determine validity
data = {}
valid = np.ones(rows.size, dtype=bool) # True where every FEATURE is good

for name, path in FILES.items():
    with rasterio.open(path) as src:
        band = src.read(1)
        nod = src.nodata
    v = band[rows, cols].astype(np.float32)
    bad = ~np.isfinite(v)     # catches inf
    if nod is not None:
        bad |= (v== np.float32(nod)) # catches -9999 and/or -3.4e38 nodata
    v[bad] = np.nan         # fill nodata with NaN
    data[name] = v
    if name != "ddem":      # target may be NaN; features may not be
        valid &= ~bad     # bitwise boolean: lhs assigned True only if both are True

has_t = ~np.isnan(data["ddem"])

print("slope pixels:", rows.size,
      "| all features valid:", int(valid.sum()),
      "| valid with target:", int((valid & has_t).sum()))

slope pixels: 2605575 | all features valid: 2575353 | valid with target: 2575353


In [4]:
# 3. tabulize
import pandas as pd
keep = valid
r = rows[keep]; c = cols[keep]
sidk = sid[r, c].astype(np.int8)  # valid slope ids

gl = np.array(["canwell","canwell","castner","castner","fels","fels"])
sd = np.array(["N","S","N","S","N","S"])

# each pixel was assigned glacier index 1-3 and side index 1 or 2; adjusting each by -1 to python indexing 
glacier = gl[sidk - 1]; side = sd[sidk - 1] 

# retreiving cell-center grid (global) coordinates
x = transform.c + transform.a*(c + 0.5) # cell-center easting
y = transform.f + transform.e*(r + 0.5) # cell-center northing

asp = np.deg2rad(data["aspect"][keep])

df = pd.DataFrame({
    "slope_id": sidk, "glacier":glacier, "side":side,
    "row": r.astype(np.int32), "col": c.astype(np.int32), "x":x, "y":y,
    "elev": data["elev"][keep], "slope": data["slope"][keep],
    "aspect_sin": np.sin(asp).astype(np.float32),
    "aspect_cos": np.cos(asp).astype(np.float32),
    "roughness": data["roughness"][keep], "tri": data["tri"][keep],
    "tpi": data["tpi"][keep], "ndvi": data["ndvi"][keep],
    "lst": data["lst"][keep], "dist": data["dist"][keep],
    "ddem": data["ddem"][keep]
})

print(df.shape)
print(df.groupby(["glacier", "side"]).size())
print(df[["elev","slope","ndvi","lst","dist","ddem"]].describe().round(2))


(2575353, 18)
glacier  side
canwell  N       477995
         S       253032
castner  N       548756
         S       454097
fels     N       639999
         S       201474
dtype: int64
             elev       slope        ndvi         lst        dist        ddem
count  2575353.00  2575353.00  2575353.00  2575353.00  2575353.00  2575353.00
mean      1396.73       26.44       -0.07       11.72      669.77       -0.11
std        271.47        9.60        0.12        9.84      499.65        3.22
min        801.94        0.06       -0.45       -6.22        5.00      -53.34
25%       1194.72       20.07       -0.10        1.93      261.01       -1.13
50%       1382.13       26.60       -0.02       10.68      559.13        0.02
75%       1579.49       32.72        0.00       20.34      984.59        1.20
max       2185.41       83.43        0.30       35.16     2219.76       66.39


In [5]:
# 4. save table + column roles
import json, datetime
out = "../data/constructs1/slope_feature_table.parquet"
df.to_parquet(out, index=False) # needs pyarrow

roles = {
    "created": datetime.date.today().isoformat(),
    "n_rows": int(df.shape[0]),
    "grid": "5965x5773 @5m EPSG:32606",
    "features": ["elev","slope","aspect_sin","aspect_cos","roughness",
                 "tri","tpi","ndvi","lst","dist"],
    "target": "ddem",
    "tags": ["slope_id","glacier","side"],
    "coords": ["row","col","x","y"],
    "slope_id_map": {1:"canwell-N",2:"canwell-S",3:"castner-N",
                     4:"castner-S",5:"fels-N",6:"fels-S"},
    "notes": "aspect split to sin/cos; dist=metres to nearest basin; ddem high extreme values",
}

with open("../data/constructs1/slope_feature_table.meta.json","w") as f:
    json.dump(roles, f, indent=2)
print("saved", out, "|", df.shape, "| features:", len(roles["features"]))

saved ../data/constructs1/slope_feature_table.parquet | (2575353, 18) | features: 10
